# Feast Feature Store on prokube

**Scenario:** You work at an online retailer. Your team wants to predict
whether a customer will return their next order. To do that, you need
customer-level features (order history, return rates, spending patterns)
available for both model training and real-time inference.

This notebook walks through the full Feast workflow:

1. **Setup**: install Feast, start Redis in your workspace, configure the client
2. **Generate data**: simulate customer order history
3. **Define features**: entities, feature views, and on-demand transformations
4. **Register**: write the definitions to the Feast registry
5. **Train**: retrieve historical features and train a return predictor
6. **Materialize**: push the latest values to Redis for online serving
7. **Serve**: predict return risk for incoming orders in real time

Feast runs as a Python library in this notebook. prokube does not need a Feast
server or operator, and no administrator action is required.

### Prerequisites

- A prokube Lab (JupyterLab) in your workspace, with this repository cloned
- Open this notebook from the `feast/` directory

Run all cells in order. See `README.md` for what the notebook creates and how
to remove it.

---
## 1. Setup

In [ ]:
%pip install -q 'feast[aws,redis]==0.68.0' scikit-learn

### Infrastructure setup

The cell below creates everything Feast needs in your workspace. It is safe to
re-run: existing resources are kept.

| Step | What it does |
|------|--------------|
| 1 | Creates the `feast-redis` secret with a random Redis password |
| 2 | Applies `redis.yaml` (a single Redis StatefulSet and Service) and waits until it is ready |
| 3 | Checks that this notebook can connect to Redis |

Feature definitions and training data are stored in your workspace bucket
`<workspace>-data` under the `feast-example/` prefix.

In [ ]:
import base64
import os
import secrets
import subprocess

import redis


def get_namespace():
    with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
        return f.read().strip()


def kubectl(*args):
    return subprocess.run(["kubectl", *args], check=True, capture_output=True, text=True).stdout


if not os.path.exists("redis.yaml"):
    raise RuntimeError("redis.yaml not found. Open this notebook from the feast/ directory.")

NAMESPACE = get_namespace()
BUCKET = f"{NAMESPACE}-data"
S3_ROOT = f"s3://{BUCKET}/feast-example"
print(f"Namespace: {NAMESPACE}")
print(f"Storage:   {S3_ROOT}")

# 1. Redis password secret
if subprocess.run(["kubectl", "get", "secret", "feast-redis"], capture_output=True).returncode != 0:
    kubectl(
        "create", "secret", "generic", "feast-redis",
        f"--from-literal=password={secrets.token_hex(24)}",
    )
    print("Created secret feast-redis")
REDIS_PASSWORD = base64.b64decode(
    kubectl("get", "secret", "feast-redis", "-o", "jsonpath={.data.password}")
).decode()

# 2. Redis StatefulSet and Service
kubectl("apply", "-f", "redis.yaml")
print("Waiting for Redis...")
kubectl("rollout", "status", "statefulset/feast-redis", "--timeout=300s")

# 3. Connectivity check
try:
    redis.Redis(
        host="feast-redis", port=6379, password=REDIS_PASSWORD,
        socket_connect_timeout=10, socket_timeout=10,
    ).ping()
except redis.exceptions.ConnectionError as exc:
    raise RuntimeError(
        "Redis is running but not reachable from this notebook. If your workspace has an "
        "egress profile, apply redis-networkpolicy.yaml and re-run this cell."
    ) from exc
print("Redis is ready.")

### Configure the Feast client

`feature_store.yaml` in this directory tells Feast where its three stores live:

| Store | Purpose | Backend in this example |
|-------|---------|-------------------------|
| **Registry** | Feature definitions, written by `apply()` | File in your workspace bucket |
| **Offline store** | Historical feature data for training | Parquet files in your workspace bucket |
| **Online store** | Latest value per entity, read at inference time | Redis in your workspace |

The file references two environment variables instead of hard-coded values, so
the Redis password never ends up in a file. Re-run this cell after a kernel
restart.

In [ ]:
os.environ["FEAST_REGISTRY_PATH"] = f"{S3_ROOT}/registry.db"
os.environ["FEAST_REDIS_CONNECTION_STRING"] = f"feast-redis:6379,password={REDIS_PASSWORD}"

print(open("feature_store.yaml").read())

---
## 2. Generate sample data

We simulate a customer order history table, the kind of data your data
pipeline would produce daily. Each row represents the aggregated stats for
one customer at one point in time:

| Column | Meaning |
|--------|---------|
| `customer_id` | Unique customer identifier |
| `total_orders` | Total number of orders placed |
| `total_returns` | Total number of returned orders |
| `avg_order_value` | Average order value in EUR |
| `days_since_last_order` | Days since the customer's last order |
| `returned` | Did the customer return their most recent order? (label) |

In a real project, this would come from your data warehouse or ETL pipeline.

In [ ]:
import datetime

import numpy as np
import pandas as pd

np.random.seed(42)
n_customers = 200
n_snapshots = 10  # 10 daily snapshots per customer
n = n_customers * n_snapshots
now = datetime.datetime.now(datetime.timezone.utc)

customer_ids = np.repeat(np.arange(1, n_customers + 1), n_snapshots)
timestamps = []
for _ in range(n_customers):
    timestamps.extend([now - datetime.timedelta(days=i) for i in range(n_snapshots)])

# Simulate realistic customer stats
total_orders = np.random.randint(1, 80, n).astype(np.int64)
total_returns = np.array([
    np.random.binomial(orders, np.random.uniform(0.05, 0.4))
    for orders in total_orders
]).astype(np.int64)
avg_order_value = np.random.uniform(15.0, 250.0, n).astype(np.float32)
days_since_last_order = np.random.randint(0, 90, n).astype(np.int64)

# Label: customers with high return rates and high order values are more
# likely to return. Add noise to keep it realistic.
return_rate = total_returns / np.maximum(total_orders, 1)
return_prob = 0.3 * return_rate + 0.002 * avg_order_value / 250.0 + np.random.normal(0, 0.1, n)
returned = (return_prob > 0.15).astype(np.int64)

customer_df = pd.DataFrame({
    "customer_id": customer_ids,
    "event_timestamp": timestamps,
    "total_orders": total_orders,
    "total_returns": total_returns,
    "avg_order_value": avg_order_value,
    "days_since_last_order": days_since_last_order,
    "returned": returned,
    "created": timestamps,
})

DATA_PATH = f"{S3_ROOT}/data/customer_orders.parquet"
customer_df.to_parquet(DATA_PATH)
print(f"Wrote {n} rows for {n_customers} customers to {DATA_PATH}")
print(f"Return rate in dataset: {returned.mean():.1%}")
customer_df.head(10)

---
## 3. Define features

In Feast, features are defined as Python objects:

- **Entity**: the primary key for lookups (here, `customer_id`).
- **DataSource**: where the raw feature data lives (here, the Parquet file in
  your bucket).
- **FeatureView**: declares which columns from the source are features and
  how long they are valid (`ttl`). Feast stores and serves them but does not
  compute them. Your data pipeline produces the data.
- **On-demand feature view**: derived features that Feast computes at
  retrieval time, both for training and for online serving.
- **FeatureService**: a named bundle of feature views. Consumers such as an
  inference service reference the service by name instead of listing
  individual features.

In [ ]:
from datetime import timedelta

from feast import Entity, FeatureService, FeatureStore, FeatureView, Field, FileSource, ValueType
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Int64

# Entity: the "primary key" for feature lookups.
customer = Entity(
    name="customer",
    join_keys=["customer_id"],
    value_type=ValueType.INT64,
    description="Unique customer identifier",
)

# Data source: the Parquet file written above. Feast reads s3:// sources with
# PyArrow, which needs the endpoint passed explicitly.
customer_orders_source = FileSource(
    name="customer_orders_source",
    path=DATA_PATH,
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
    s3_endpoint_override=os.environ["AWS_ENDPOINT_URL"],
)

# FeatureView: raw, precomputed values from the data pipeline.
# "returned" is the prediction target, so it is not a feature: at serving
# time the answer is unknown. Labels are joined only during training.
customer_order_stats = FeatureView(
    name="customer_order_stats",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="total_orders", dtype=Int64),
        Field(name="total_returns", dtype=Int64),
        Field(name="avg_order_value", dtype=Float32),
        Field(name="days_since_last_order", dtype=Int64),
    ],
    source=customer_orders_source,
    online=True,
)


# On-demand feature view: Feast computes these during
# get_historical_features() and get_online_features(), so training and
# serving use the same logic.
@on_demand_feature_view(
    sources=[customer_order_stats],
    schema=[
        Field(name="return_rate", dtype=Float32),
        Field(name="return_risk", dtype=Float32),
    ],
)
def customer_risk_features(inputs: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame()
    df["return_rate"] = (
        inputs["total_returns"] / inputs["total_orders"].clip(lower=1)
    ).astype("float32")
    df["return_risk"] = (df["return_rate"] * inputs["avg_order_value"]).astype("float32")
    return df


# FeatureService: one name for everything the model needs.
customer_risk_service = FeatureService(
    name="customer_risk_service",
    features=[customer_order_stats, customer_risk_features],
)

print("Feature definitions created (not yet registered).")

---
## 4. Register features

`store.apply()` writes all definitions to the registry file in your bucket.
Any notebook, pipeline step, or service in the workspace that uses the same
`feature_store.yaml` sees the same definitions.

You only need to re-run this cell if you change a definition.

In [ ]:
store = FeatureStore(repo_path=".")

store.apply([
    customer,
    customer_orders_source,
    customer_order_stats,
    customer_risk_features,
    customer_risk_service,
])

print(f"Registered in {os.environ['FEAST_REGISTRY_PATH']}:")
for fv in store.list_feature_views():
    print(f"  FeatureView:    {fv.name}")
for odfv in store.list_on_demand_feature_views():
    print(f"  OnDemandFV:     {odfv.name}")
for fs in store.list_feature_services():
    print(f"  FeatureService: {fs.name}")

---
## 5. Retrieve historical features and train a model

`get_historical_features()` performs a **point-in-time join**: for each
customer, it finds the most recent feature values *as of the requested
timestamp*. This prevents data leakage: you only see features that were
available when the event occurred.

The feature service includes the on-demand feature view, so `return_rate` and
`return_risk` are computed by Feast as part of the same call.

In [ ]:
# "Give me features for these customers, as of right now."
all_customer_ids = list(range(1, n_customers + 1))

entity_df = pd.DataFrame({
    "customer_id": all_customer_ids,
    "event_timestamp": [now] * len(all_customer_ids),
})

print(f"Querying features for {len(all_customer_ids)} customers...")

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=customer_risk_service,
).to_df()

print(f"Retrieved {len(training_df)} rows.")
training_df.head(10)

### Preprocessing

Before training, we clean up the data:

1. **Join the label.** `returned` is the prediction target, not a feature. It
   stays out of Feast because at serving time (when a customer places a new
   order) the answer is unknown. Labels usually come from a separate source
   and are joined only for training.
2. **Filter out new customers.** Customers with fewer than 3 orders don't have
   enough history for reliable features.
3. **Drop nulls.** Rows where Feast found no matching features.
4. **Normalize.** Scale numeric features so the model doesn't overweight
   high-magnitude columns like `avg_order_value`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Join the label: most recent snapshot per customer
latest_labels = (
    customer_df
    .sort_values("event_timestamp")
    .groupby("customer_id")
    .last()[["returned"]]
    .reset_index()
)
training_df = training_df.merge(latest_labels, on="customer_id", how="left")
print(f"After joining labels: {len(training_df)} rows")

# 2. Filter out new customers (< 3 orders)
before = len(training_df)
training_df = training_df[training_df["total_orders"] >= 3].copy()
print(f"After filtering new customers: {len(training_df)} rows (dropped {before - len(training_df)})")

# 3. Drop nulls
before = len(training_df)
training_df = training_df.dropna()
print(f"After dropping nulls: {len(training_df)} rows (dropped {before - len(training_df)})")

# 4. Normalize features
FEATURE_COLS = ["total_orders", "total_returns", "avg_order_value",
                "days_since_last_order", "return_rate", "return_risk"]
TARGET = "returned"

scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(training_df[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=training_df.index,
)
y = training_df[TARGET]

print(f"\nTraining set: {len(X)} samples, {len(FEATURE_COLS)} features")
print(f"Class balance: {y.mean():.1%} returns")
X.describe().round(2)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["kept", "returned"]))

---
## 6. Materialize features to Redis

Materialization copies the latest feature values from the offline store
(Parquet) into Redis for low-latency online serving.

Only `FeatureView` data is written to Redis. The on-demand features
(`return_rate`, `return_risk`) are computed when you call
`get_online_features()`.

In production, run this on a schedule, for example as a recurring pipeline run
after your data pipeline updates the Parquet files.

In [ ]:
store.materialize(
    start_date=now - timedelta(days=30),
    end_date=now,
)
print("Materialized to Redis.")

---
## 7. Online feature serving: predict return risk

A customer just placed an order. Your order service asks Feast for their
features, which Feast reads from Redis, applies the on-demand transformation,
and returns. The model then decides whether to flag the order for proactive
customer service.

In [ ]:
# These 3 customers just placed a new order
customers_with_new_orders = [{"customer_id": 5}, {"customer_id": 42}, {"customer_id": 137}]

online_features = store.get_online_features(
    features=customer_risk_service,
    entity_rows=customers_with_new_orders,
).to_dict()

online_df = pd.DataFrame(online_features)

print("Online features (raw values from Redis + on-demand derived columns):")
online_df

In [ ]:
X_inference = pd.DataFrame(
    scaler.transform(online_df[FEATURE_COLS]),
    columns=FEATURE_COLS,
)

return_probabilities = model.predict_proba(X_inference)[:, 1]

RISK_THRESHOLD = 0.5

print("--- Return Risk Assessment ---")
for cid, prob in zip(online_df["customer_id"], return_probabilities):
    flag = "HIGH RISK" if prob > RISK_THRESHOLD else "low risk"
    print(f"  Customer {cid}: return probability = {prob:.1%} [{flag}]")

---
## Summary

| Step | API | What happens |
|------|-----|--------------|
| Define | Python objects (Entity, FeatureView, FeatureService) | Declare what features exist and how they are grouped |
| Register | `store.apply([...])` | Write definitions to the registry file in your bucket |
| Train | `store.get_historical_features(features=service)` | Point-in-time join from Parquet, plus on-demand features |
| Materialize | `store.materialize()` | Push the latest raw values to Redis |
| Serve | `store.get_online_features(features=service)` | Low-latency lookup from Redis, plus on-demand features |

### Use the features from another workload

Any pod in this workspace, such as a pipeline step or an inference service, can
read the same features. It needs:

- Feast installed and this directory's `feature_store.yaml`;
- `FEAST_REGISTRY_PATH` and `FEAST_REDIS_CONNECTION_STRING` set as in the
  configuration cell;
- S3 credentials and `AWS_ENDPOINT_URL`, which Labs receive automatically.

`README.md` shows the environment variables for a pod manifest.

## Toward production

| Concern | This notebook | Production |
|---------|---------------|------------|
| Feature definitions | Defined in the notebook | Python files in Git; `feast apply` runs in CI or a pipeline step |
| Materialization | Run from the notebook | Recurring pipeline run or CronJob |
| Registry | File in the workspace bucket | Same, or a SQL registry when many writers run `apply` concurrently |
| Online store | Single Redis pod in the workspace | Replicated Redis or PostgreSQL provided by your administrator |
| Offline store | Parquet in the workspace bucket | Same, or a warehouse supported by Feast |

See [Feast's production guide](https://docs.feast.dev/how-to-guides/running-feast-in-production).

---
## Clean up

To remove what this notebook created, run these commands in a terminal from
the `feast/` directory:

```bash
kubectl delete -f redis.yaml --ignore-not-found
kubectl delete pvc data-feast-redis-0 --ignore-not-found
kubectl delete secret feast-redis --ignore-not-found
rclone purge minio:<workspace>-data/feast-example
```